In [ ]:
from IPython.display import Markdown, display
from yandex_cloud_ml_sdk import YCloudML
from yandex_cloud_ml_sdk.search_indexes import (
    StaticIndexChunkingStrategy,
    HybridSearchIndexType,
    ReciprocalRankFusionIndexCombinationStrategy,
)
from typing import Optional
from pydantic import BaseModel, Field

folder_id = 'b1gst3c7cskk2big5fqn'
api_key = 'AQVNzzJielnSayrAOlQWlxDMK49OShvzdqtUQdAp'
sdk = YCloudML(folder_id=folder_id, auth=api_key)
model = sdk.models.completions("yandexgpt", model_version="rc")

class GetAdmissionDeadlineParams(BaseModel):
    """Функция возвращает крайний срок подачи документов в МАИ."""

def get_admission_deadline(**kwargs):
    class ToolResponse:
        def process(self, thread):
            return "Крайний срок подачи документов в МАИ — 25 июля 2025 года."
    return ToolResponse()

class CallOperator(BaseModel):
    """Функция возвращает крайний срок подачи документов в МАИ."""

def get_call_operator(**kwargs):
    class ToolResponse:
        def process(self, thread):
            return "вызов оператора"
    return ToolResponse()

def printx(string):
    display(Markdown(string))

def upload_file():
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    return sdk.files.upload(filename, ttl_days=1, expiration_policy="static")

def create_thread():
    return sdk.threads.create(ttl_days=1, expiration_policy="static")

def create_assistant(model, tools=None):
    kwargs = {}
    if tools and len(tools) > 0:
        kwargs = {"tools": tools}
    return sdk.assistants.create(
        model, ttl_days=1, expiration_policy="since_last_active", **kwargs
    )

class Agent:
    def __init__(self, assistant=None, instruction=None, search_index=None, tools=None):
        self.thread = None
        if assistant:
            self.assistant = assistant
        else:
            if tools:
                self.tools = {x['name']: x['fn'] for x in tools}
                tool_defs = [sdk.tools.function(x['model']) for x in tools]
            else:
                self.tools = {}
                tool_defs = []
            if search_index:
                tool_defs.append(sdk.tools.search_index(search_index))
            self.assistant = create_assistant(model, tool_defs)
        if instruction:
            self.assistant.update(instruction=instruction)

    def get_thread(self, thread=None):
        if thread:
            return thread
        if self.thread is None:
            self.thread = create_thread()
        return self.thread

    def __call__(self, message, thread=None):
        thread = self.get_thread(thread)
        thread.write(message)
        run = self.assistant.run(thread)
        res = run.wait()
        if res.tool_calls:
            result = []
            for f in res.tool_calls:
                print(f" + Вызов функции: {f.function.name}, args={f.function.arguments}")
                fn = self.tools[f.function.name]
                obj = fn(**f.function.arguments)
                x = obj.process(thread)
                result.append({"name": f.function.name, "content": x})
            run.submit_tool_results(result)
            res = run.wait()
        return res.text

    def restart(self):
        if self.thread:
            self.thread.delete()
            self.thread = sdk.threads.create(name="Test", ttl_days=1, expiration_policy="static")

    def done(self, delete_assistant=False):
        if self.thread:
            self.thread.delete()
        if delete_assistant:
            self.assistant.delete()

def upload_file():
    return sdk.files.upload('/Users/ogzeus/Downloads/data2023.json', ttl_days=1, expiration_policy="static")
g = upload_file()

op = sdk.search_indexes.create_deferred(
    g,
    index_type=HybridSearchIndexType(
        chunking_strategy=StaticIndexChunkingStrategy(
            max_chunk_size_tokens=1000,
            chunk_overlap_tokens=100
        ),
        combination_strategy=ReciprocalRankFusionIndexCombinationStrategy(),
    ),
)
index = op.wait()

In [ ]:
instruction = """
Ты — сотрудник приёмной комиссии МАИ. Отвечай только на вопросы по поступлению.
Если уместно — используй функцию для вызова оператора и вызови только функцию CallOperator.
Если уместно — используй функцию для определения крайнего срока подачи документов.
Если пользователь не может получить ответ или , то напомни ему что он может вызвать оператора
"""
agent = Agent(
    instruction=instruction,
    search_index=index,
    tools=[{
        "name": "GetAdmissionDeadlineParams",
        "model": GetAdmissionDeadlineParams,
        "fn": get_admission_deadline
    }, {
        "name": "CallOperator",
        "model": CallOperator,
        "fn": get_call_operator
    }]
)

response = agent("я не получил ответ на свой вопрос")
printx(response)

In [ ]:
instruction = """
Ты — сотрудник приёмной комиссии МАИ. Отвечай только на вопросы по поступлению.
Если уместно — используй функцию для вызова оператора и вызови только функцию CallOperator.
Если уместно — используй функцию для определения крайнего срока подачи документов.
Если пользователь не может получить ответ или , то напомни ему что он может вызвать оператора
"""
agent = Agent(
    instruction=instruction,
    search_index=index,
    tools=[{
        "name": "GetAdmissionDeadlineParams",
        "model": GetAdmissionDeadlineParams,
        "fn": get_admission_deadline
    }, {
        "name": "CallOperator",
        "model": CallOperator,
        "fn": get_call_operator
    }]
)

response = agent("я не получил ответ на свой вопрос")
printx(response)